In [19]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


In [24]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Constants
ROOT_DIR = r"E:\Upwork Project\AI_Leak_Detection_Project\images\cwt_log"
SENSORS = ["Accelerometer", "Dynamic Pressure Sensor", "Hydrophones"]
CLASSES = ["No-leak", "Orifice Leak", "Gasket Leak", "Longitudinal Crack", "Circumferential Crack"]

# Preprocessing
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
NUM_WORKERS = 0

# Dataset Class
class SensorStructuredCWTDataset(Dataset):
    def __init__(self, root_dir, sensors, class_names, transform=None):
        self.root_dir = root_dir
        self.sensors = sensors
        self.class_names = class_names
        self.transform = transform
        self.samples = []

        for class_idx, class_name in enumerate(class_names):
            # Collect all filenames from the first sensor folder as reference
            ref_folder = os.path.join(root_dir, sensors[0], "Looped", class_name)
            if not os.path.isdir(ref_folder):
                print(f"[!] Missing reference folder: {ref_folder}")
                continue

            filenames = sorted(os.listdir(ref_folder))

            for filename in filenames:
                img_paths = []
                valid_sample = True

                for sensor in sensors:
                    sensor_path = os.path.join(root_dir, sensor, "Looped", class_name, filename)
                    if not os.path.isfile(sensor_path):
                        valid_sample = False
                        break
                    img_paths.append(sensor_path)

                if valid_sample and len(img_paths) == 3:
                    self.samples.append((img_paths, class_idx))

        print(f"[SUMMARY] Total valid samples: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_paths, label = self.samples[idx]
        imgs = []
        for path in img_paths:
            image = Image.open(path).convert("L")
            if self.transform:
                image = self.transform(image)
            imgs.append(image)
        stacked = torch.stack(imgs, dim=0)  # (3, H, W)
        return stacked, label


In [26]:
import os

ROOT_DIR = r"E:\Upwork Project\AI_Leak_Detection_Project\images\cwt_log"
SENSORS = ["Accelerometer", "Dynamic Pressure Sensor", "Hydrophones"]
CLASSES = ["No-leak", "Orifice Leak", "Gasket Leak", "Longitudinal Crack", "Circumferential Crack"]

for sensor in SENSORS:
    print(f"\n--- Sensor: {sensor} ---")
    for class_name in CLASSES:
        folder = os.path.join(ROOT_DIR, sensor, "Looped", class_name)
        if os.path.isdir(folder):
            files = sorted(os.listdir(folder))
            print(f"  {class_name} ({len(files)} files): {files[:5]}")
        else:
            print(f"  [!] MISSING FOLDER: {folder}")



--- Sensor: Accelerometer ---
  No-leak (250 files): ['Accelerometer_No-leak_001.png', 'Accelerometer_No-leak_002.png', 'Accelerometer_No-leak_003.png', 'Accelerometer_No-leak_004.png', 'Accelerometer_No-leak_005.png']
  Orifice Leak (250 files): ['Accelerometer_Orifice_Leak_001.png', 'Accelerometer_Orifice_Leak_002.png', 'Accelerometer_Orifice_Leak_003.png', 'Accelerometer_Orifice_Leak_004.png', 'Accelerometer_Orifice_Leak_005.png']
  Gasket Leak (250 files): ['Accelerometer_Gasket_Leak_001.png', 'Accelerometer_Gasket_Leak_002.png', 'Accelerometer_Gasket_Leak_003.png', 'Accelerometer_Gasket_Leak_004.png', 'Accelerometer_Gasket_Leak_005.png']
  Longitudinal Crack (250 files): ['Accelerometer_Longitudinal_Crack_001.png', 'Accelerometer_Longitudinal_Crack_002.png', 'Accelerometer_Longitudinal_Crack_003.png', 'Accelerometer_Longitudinal_Crack_004.png', 'Accelerometer_Longitudinal_Crack_005.png']
  Circumferential Crack (250 files): ['Accelerometer_Circumferential_Crack_001.png', 'Acceler

In [ ]:
import re
from PIL import Image
from torch.utils.data import Dataset
import torch
import os

class FixedPatternCWTDataset(Dataset):
    def __init__(self, root_dir, sensors, class_names, transform=None):
        self.root_dir = root_dir
        self.sensors = sensors
        self.class_names = class_names
        self.transform = transform
        self.samples = []

        for class_idx, class_name in enumerate(class_names):
            sample_ids = range(1, 251)  # You have 250 samples per class

            for sample_id in sample_ids:
                sample_str = f"{sample_id:03d}"
                sensor_paths = []

                for sensor in sensors:
                    class_folder = os.path.join(root_dir, sensor, "Looped", class_name)

                    if not os.path.isdir(class_folder):
                        print(f"[!] Missing folder: {class_folder}")
                        break

                    if sensor == "Accelerometer":
                        filename = f"Accelerometer_{class_name.replace(' ', '_')}_{sample_str}.png"
                    elif sensor == "Dynamic Pressure Sensor":
                        filename = f"DynamicPressureSensor_{class_name.replace(' ', '_')}_{sample_str}.png"
                    elif sensor == "Hydrophones":
                        filename = f"{class_name.replace(' ', '_')}_segment_{sample_str}.png"
                    else:
                        raise ValueError("Unknown sensor type")

                    full_path = os.path.join(class_folder, filename)
                    if not os.path.isfile(full_path):
                        print(f"[!] Missing file: {full_path}")
                        break

                    sensor_paths.append(full_path)

                if len(sensor_paths) == 3:
                    self.samples.append((sensor_paths, class_idx))

        print(f"[SUMMARY] Total valid samples: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        paths, label = self.samples[idx]
        images = []
        for path in paths:
            image = Image.open(path).convert("L")
            if self.transform:
                image = self.transform(image)
            images.append(image)
        stacked = torch.cat(imgs, dim=0)   # Shape: (3, H, W)
        return stacked, label



In [28]:
from torchvision import transforms
from torch.utils.data import DataLoader

# Constants
ROOT_DIR = r"E:\Upwork Project\AI_Leak_Detection_Project\images\cwt_log"
SENSORS = ["Accelerometer", "Dynamic Pressure Sensor", "Hydrophones"]
CLASSES = ["No-leak", "Orifice Leak", "Gasket Leak", "Longitudinal Crack", "Circumferential Crack"]

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

dataset = FixedPatternCWTDataset(ROOT_DIR, SENSORS, CLASSES, transform=transform)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0)

# Sanity check
for x, y in train_loader:
    print(f"[BATCH] x: {x.shape}, y: {y.shape}")
    break


[SUMMARY] Total valid samples: 1250
[BATCH] x: torch.Size([32, 3, 1, 128, 128]), y: torch.Size([32])
